### Import Dependencies

In [1]:
import openai
import instructor
from pydantic import BaseModel, Field
import pandas as pd

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import Filter, FieldCondition, MatchValue, VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery, MatchAny

import numpy as np
import tiktoken

### Retrieve all item IDs from Amazon Items Qdrant Collection

In [3]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [4]:
dummy_vector = np.zeros(1536).tolist()

In [5]:
payload = qdrant_client.query_points(
    collection_name="Amazon-items-collection-01-hybrid-search",
    query=dummy_vector,
    using="text-embedding-3-small",
    limit=1000,
    with_payload=["parent_asin"],
    with_vectors=False
)

In [6]:
payload.points

[ScoredPoint(id=332, version=3, score=0.0, payload={'parent_asin': 'B09D3GRST4'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=162, version=3, score=0.0, payload={'parent_asin': 'B09YB3ZMRT'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=175, version=3, score=0.0, payload={'parent_asin': 'B09WCFC5D9'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=285, version=3, score=0.0, payload={'parent_asin': 'B09SHHC38J'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=86, version=3, score=0.0, payload={'parent_asin': 'B09R4JB5Q3'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=51, version=3, score=0.0, payload={'parent_asin': 'B0BM963XNT'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=449, version=3, score=0.0, payload={'parent_asin': 'B09MWJJ3RW'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=490, version=3, score=0.0, payload={'parent_asin': 'B0B24RXLGY'}, vector=Non

In [7]:
len(payload.points)

1000

In [8]:
parent_asin_list = [item.payload["parent_asin"] for item in payload.points]

In [9]:
parent_asin_list

['B09D3GRST4',
 'B09YB3ZMRT',
 'B09WCFC5D9',
 'B09SHHC38J',
 'B09R4JB5Q3',
 'B0BM963XNT',
 'B09MWJJ3RW',
 'B0B24RXLGY',
 'B09Y77N1T2',
 'B0CHGXHB81',
 'B0B87Y9XXB',
 'B0BF42MF7K',
 'B09MRBYHKG',
 'B0C7Q3X76Q',
 'B0BBY5TGTM',
 'B0C3QYJZN5',
 'B08CGM6Y1J',
 'B0BYKDMMVJ',
 'B0BW3GT2YJ',
 'B09YHWCNCH',
 'B0B6R8HNH6',
 'B0C13NF5JC',
 'B09YTGFYZK',
 'B0BNKJDF99',
 'B0BNPVBZPG',
 'B09R6STLP7',
 'B09PYFMTBF',
 'B0BML99MQ3',
 'B0C78B1BTB',
 'B0BZVHY74B',
 'B0C3XS6N81',
 'B0BPNNV5NW',
 'B0CG57QKZ8',
 'B09QS7W8G5',
 'B0BJV3VJC6',
 'B0B45VPMT4',
 'B09Q9C6MG1',
 'B0BW89VYWN',
 'B09Q88GR4K',
 'B0BF9DQB6K',
 'B0BL3YGC62',
 'B0BXQ4R99F',
 'B0BK14FQMQ',
 'B09GNFDH9M',
 'B09W5RJ1LY',
 'B0B39PFLSW',
 'B0BJTBFY5B',
 'B0BYN7PTKX',
 'B09VT7PW8K',
 'B09STGSRW2',
 'B09ZV2XJ65',
 'B09NPP7Z6G',
 'B08HR18MRH',
 'B0BG2SKNPM',
 'B09QS2D9TZ',
 'B0C6Q65325',
 'B0973DV11T',
 'B09X2X2BKC',
 'B09PQNT9F3',
 'B0B6PKDSMN',
 'B0BQJWKBMF',
 'B0BCF5DLWV',
 'B0BGMYY12Z',
 'B0BMX2ZGKX',
 'B0BVH2K81J',
 'B0BN2VHZV2',
 'B0C996WY

### Load Amazon Reviews Dataset

In [11]:
df_reviews = pd.read_json("../../data/Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [12]:
df_reviews.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5,Perfect!,This is perfect! Thank you so much!!! I absolu...,[],B09992M2LX,B09ZPV8WBV,AHX4XWVVQUKT3FCNWCVASDF4Q56Q,2022-08-05 04:06:39.589,0,True
1,5,3ft mini usb cables,I don't have many things that still use a mini...,[],B09Y94B2NM,B09Y95BMKX,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2022-07-16 16:03:28.714,3,True
2,5,I would buy it again.,Great product. Worked well for what we needed ...,[],B07T55DL33,B0B2JWCMCY,AF5KFHNT3TQJ2GNSE3FCDFQOBICA,2019-12-09 22:35:00.531,0,True
3,5,Great to Have Around,My husband and I were recently working a booth...,[],B09M89JN7B,B0BYYGZHG5,AHV6QCNBJNSGLATP56JAWJ3C4G2A,2022-03-22 01:43:49.342,0,False
4,5,Easy to use,Work as advertised and at a very good price.,[],B07T55DL33,B0B2JWCMCY,AG7WKTZINOFIXMZJYIPKIB7PV7NQ,2019-12-28 06:12:24.960,0,True


In [13]:
len(df_reviews)

105918

In [14]:
df_reviews_sample = df_reviews[df_reviews["parent_asin"].isin(parent_asin_list)]

In [15]:
len(df_reviews_sample)

105918

### Define functions to preprocess reviews data

In [16]:
def preprocess_reviews_data(row):
    return f"{row['title']} {row['text']}"

In [17]:
encoding = tiktoken.encoding_for_model("text-embedding-3-small")

In [18]:
encoding.encode("Can I get some earphones?")

[6854, 358, 636, 1063, 2487, 17144, 30]

In [19]:
def token_count(row, model="text-embedding-3-small"):

    encoding = tiktoken.encoding_for_model(model)

    return len(encoding.encode(row["preprocessed_data"]))

In [20]:
df_reviews_sample["preprocessed_data"] = df_reviews_sample.apply(preprocess_reviews_data, axis=1)
df_reviews_sample["token_count"] = df_reviews_sample.apply(token_count, axis=1)

In [21]:
df_reviews_sample.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,preprocessed_data,token_count
0,5,Perfect!,This is perfect! Thank you so much!!! I absolu...,[],B09992M2LX,B09ZPV8WBV,AHX4XWVVQUKT3FCNWCVASDF4Q56Q,2022-08-05 04:06:39.589,0,True,Perfect! This is perfect! Thank you so much!!!...,21
1,5,3ft mini usb cables,I don't have many things that still use a mini...,[],B09Y94B2NM,B09Y95BMKX,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2022-07-16 16:03:28.714,3,True,3ft mini usb cables I don't have many things t...,114
2,5,I would buy it again.,Great product. Worked well for what we needed ...,[],B07T55DL33,B0B2JWCMCY,AF5KFHNT3TQJ2GNSE3FCDFQOBICA,2019-12-09 22:35:00.531,0,True,I would buy it again. Great product. Worked we...,24
3,5,Great to Have Around,My husband and I were recently working a booth...,[],B09M89JN7B,B0BYYGZHG5,AHV6QCNBJNSGLATP56JAWJ3C4G2A,2022-03-22 01:43:49.342,0,False,Great to Have Around My husband and I were rec...,76
4,5,Easy to use,Work as advertised and at a very good price.,[],B07T55DL33,B0B2JWCMCY,AG7WKTZINOFIXMZJYIPKIB7PV7NQ,2019-12-28 06:12:24.960,0,True,Easy to use Work as advertised and at a very g...,13


In [22]:
len(df_reviews_sample)

105918

In [23]:
df_reviews_sample = df_reviews_sample[df_reviews_sample["token_count"] < 8192]

In [24]:
len(df_reviews_sample)

105918

In [25]:
total_tokens = df_reviews_sample["token_count"].sum()

In [26]:
total_tokens

np.int64(6204437)

### Create Qdrant colection for hybrid search

In [27]:
qdrant_client.create_collection(
    collection_name="Amazon-reviews-collection-01",
    vectors_config={
        "text-embedding-3-small": VectorParams(size=1536, distance=Distance.COSINE)
    }
)

True

In [28]:
qdrant_client.create_payload_index(
    collection_name="Amazon-reviews-collection-01",
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

### Embedding Functions

In [29]:
def get_embedding(text, model="text-embedding-3-small"):    
    response = openai.embeddings.create(
        input=text,
        model=model,
    )
    return response.data[0].embedding

In [30]:
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1
    
    return all_embeddings

### Embed the text and add additional fields to the payload of each vector for reviews

In [31]:
data_to_embed_reviews = df_reviews_sample[["preprocessed_data", "parent_asin"]].to_dict(orient="records")

In [32]:
data_to_embed_reviews

[{'preprocessed_data': 'Perfect! This is perfect! Thank you so much!!! I absolutely love it!!! It’s great quality!!!',
  'parent_asin': 'B09ZPV8WBV'},
 {'preprocessed_data': "3ft mini usb cables I don't have many things that still use a mini USB charger cable, but I after a while you have to replace the charger cables.  These seem to work well and it was noted in the ad that they are reinforced at the head area to prolong life.  We shall see - time will tell.  I never hesitate to update my reviews should new info seem useful.  I do not accept any discounts or deals that are not available to all shoppers. And my reviews are based purely on my personal experience with each item I review.",
  'parent_asin': 'B09Y95BMKX'},
 {'preprocessed_data': 'I would buy it again. Great product. Worked well for what we needed it for. Would buy it again.',
  'parent_asin': 'B0B2JWCMCY'},
 {'preprocessed_data': 'Great to Have Around My husband and I were recently working a booth at a trade show for his b

In [33]:
len(data_to_embed_reviews)

105918

In [34]:
text_to_embed_reviews = [item["preprocessed_data"] for item in data_to_embed_reviews]

In [35]:
text_to_embed_reviews

['Perfect! This is perfect! Thank you so much!!! I absolutely love it!!! It’s great quality!!!',
 "3ft mini usb cables I don't have many things that still use a mini USB charger cable, but I after a while you have to replace the charger cables.  These seem to work well and it was noted in the ad that they are reinforced at the head area to prolong life.  We shall see - time will tell.  I never hesitate to update my reviews should new info seem useful.  I do not accept any discounts or deals that are not available to all shoppers. And my reviews are based purely on my personal experience with each item I review.",
 'I would buy it again. Great product. Worked well for what we needed it for. Would buy it again.',
 'Great to Have Around My husband and I were recently working a booth at a trade show for his business. I set up a charging station with these. It was perfect! We were able to accommodate everyone who needed to charge their phone! The cables are a good length, not too short or t

In [36]:
embeddings = get_embeddings_batch(text_to_embed_reviews, batch_size=500)

Processed 500 of 105918
Processed 1000 of 105918
Processed 1500 of 105918
Processed 2000 of 105918
Processed 2500 of 105918
Processed 3000 of 105918
Processed 3500 of 105918
Processed 4000 of 105918
Processed 4500 of 105918
Processed 5000 of 105918
Processed 5500 of 105918
Processed 6000 of 105918
Processed 6500 of 105918
Processed 7000 of 105918
Processed 7500 of 105918
Processed 8000 of 105918
Processed 8500 of 105918
Processed 9000 of 105918
Processed 9500 of 105918
Processed 10000 of 105918
Processed 10500 of 105918
Processed 11000 of 105918
Processed 11500 of 105918
Processed 12000 of 105918
Processed 12500 of 105918
Processed 13000 of 105918
Processed 13500 of 105918
Processed 14000 of 105918
Processed 14500 of 105918
Processed 15000 of 105918
Processed 15500 of 105918
Processed 16000 of 105918
Processed 16500 of 105918
Processed 17000 of 105918
Processed 17500 of 105918
Processed 18000 of 105918
Processed 18500 of 105918
Processed 19000 of 105918
Processed 19500 of 105918
Proces

In [37]:
len(embeddings)

105918

In [38]:
pointstructs = []
i = 1
for embedding, data in zip(embeddings, data_to_embed_reviews):
    pointstructs.append(
        PointStruct(
            id=i,
            vector={
                "text-embedding-3-small": embedding,
            },
            payload=data
        )
    )
    i += 1

In [39]:
pointstructs[0].vector

{'text-embedding-3-small': [0.032562255859375,
  -0.038726806640625,
  -0.045623779296875,
  0.00441741943359375,
  0.00849151611328125,
  -0.0732421875,
  0.005634307861328125,
  0.003269195556640625,
  0.00119781494140625,
  -0.0208587646484375,
  0.031005859375,
  -0.0413818359375,
  -0.034881591796875,
  -0.0304107666015625,
  0.038726806640625,
  0.042724609375,
  -0.038299560546875,
  -0.0179595947265625,
  0.033538818359375,
  0.005954742431640625,
  0.058441162109375,
  -0.047393798828125,
  -0.00943756103515625,
  0.0026397705078125,
  -0.03656005859375,
  -0.026824951171875,
  -0.01114654541015625,
  -0.05389404296875,
  0.053558349609375,
  0.0066375732421875,
  -0.0117034912109375,
  -0.0056304931640625,
  -0.00202178955078125,
  -0.10748291015625,
  -0.005069732666015625,
  -0.016143798828125,
  -0.0192413330078125,
  0.0078582763671875,
  -0.027679443359375,
  -0.00693511962890625,
  -0.01412200927734375,
  0.033660888671875,
  0.0517578125,
  0.01177978515625,
  -0.03726

In [40]:
batch_size_qdrant = 100
counter = 1
for i in range(0, len(pointstructs), batch_size_qdrant):
    batch = pointstructs[i:i + batch_size_qdrant]
    qdrant_client.upsert(
        collection_name="Amazon-reviews-collection-01",
        points=batch,
        wait=True
    )
    print(f"Processed {counter * batch_size_qdrant} of {len(pointstructs)}")
    counter += 1

Processed 100 of 105918
Processed 200 of 105918
Processed 300 of 105918
Processed 400 of 105918
Processed 500 of 105918
Processed 600 of 105918
Processed 700 of 105918
Processed 800 of 105918
Processed 900 of 105918
Processed 1000 of 105918
Processed 1100 of 105918
Processed 1200 of 105918
Processed 1300 of 105918
Processed 1400 of 105918
Processed 1500 of 105918
Processed 1600 of 105918
Processed 1700 of 105918
Processed 1800 of 105918
Processed 1900 of 105918
Processed 2000 of 105918
Processed 2100 of 105918
Processed 2200 of 105918
Processed 2300 of 105918
Processed 2400 of 105918
Processed 2500 of 105918
Processed 2600 of 105918
Processed 2700 of 105918
Processed 2800 of 105918
Processed 2900 of 105918
Processed 3000 of 105918
Processed 3100 of 105918
Processed 3200 of 105918
Processed 3300 of 105918
Processed 3400 of 105918
Processed 3500 of 105918
Processed 3600 of 105918
Processed 3700 of 105918
Processed 3800 of 105918
Processed 3900 of 105918
Processed 4000 of 105918
Processed

### A function to run search against reviews on a prefiltered set of product IDs

In [41]:
def retrieve_prefiltered_reviews_data(query, parent_asins, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-reviews-collection-01",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                filter=Filter(
                    must=[
                        FieldCondition(
                            key="parent_asin",
                            match=MatchAny(
                                any=parent_asins
                            )
                        )
                    ]
                ),
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )

    return results

In [45]:
reviews = retrieve_prefiltered_reviews_data("bad quality", ["B09VDNKL4G"])

In [46]:
reviews.points

[ScoredPoint(id=82510, version=828, score=0.5, payload={'preprocessed_data': 'Fans with great pontential, but overall a half baked product. I ordered two of these fans and both were defective.<br /><br />-Fan 1 cannot stick to the PMW profile and speeds constantly surge up and down a couple of hundred RPM every few seconds. Makes annoying sounds as it constantly surges.<br /><br />-Fan two seems to have some factory fault with the bearing where the center hub sort of wobbles and makes an annoying sound like something is out of tolerance every few seconds at all RPM ranges.', 'parent_asin': 'B09VDNKL4G'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=95165, version=954, score=0.33333334, payload={'preprocessed_data': 'Maybe a bad initial batch? They are very annoyingly noisey. I was so excited to get these after watching the youtube reviews at launch. Turns out I am in the same boat as other Amazon reviewers who are experiencing a lot of periodic resonance noise that i

In [47]:
reviews = retrieve_prefiltered_reviews_data("bad quality", ["B09YD1FZDN", "B09QGN5YT6"])

In [48]:
reviews.points

[ScoredPoint(id=17829, version=181, score=0.5, payload={'preprocessed_data': 'Nice small recorder Nice small recorder. Easy to conceal or just have in pocket.  I found it hard to figure out how to playback file and instructions are not clear. Also I was expecting recorder to be voice activated but only manual recording available', 'parent_asin': 'B09YD1FZDN'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=23889, version=241, score=0.33333334, payload={'preprocessed_data': "If you need stealth, this is a solid contender. Great run time and quality recording. So, if your requirements are that your recorder is easy to carry, is small and compact, as well as easy to navigate and remove items from once plugged in..I would have to say this is a solid contender. I've used many recorders and recording apps and this standalone recorder seems to be the best for me.<br /><br />This is small, about the size of Wrigley's single stack gum packet. You've got a sliding record/stop bu

In [49]:
for point in reviews.points:
    print(point.payload["parent_asin"])
    print(point.payload["preprocessed_data"])
    print("-"*100)

B09YD1FZDN
Nice small recorder Nice small recorder. Easy to conceal or just have in pocket.  I found it hard to figure out how to playback file and instructions are not clear. Also I was expecting recorder to be voice activated but only manual recording available
----------------------------------------------------------------------------------------------------
B09YD1FZDN
If you need stealth, this is a solid contender. Great run time and quality recording. So, if your requirements are that your recorder is easy to carry, is small and compact, as well as easy to navigate and remove items from once plugged in..I would have to say this is a solid contender. I've used many recorders and recording apps and this standalone recorder seems to be the best for me.<br /><br />This is small, about the size of Wrigley's single stack gum packet. You've got a sliding record/stop button, FFW and REW, Pause, and a power button. The recorder comes with a USB cable and some little ear plugs to get you g

### Define a reviews tool

In [50]:
def retrieve_prefiltered_reviews_data(query, parent_asins, k=5):

    query_embedding = get_embedding(query)

    qdrant_client = QdrantClient(url="http://localhost:6333")

    results = qdrant_client.query_points(
        collection_name="Amazon-reviews-collection-01",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                filter=Filter(
                    must=[
                        FieldCondition(
                            key="parent_asin",
                            match=MatchAny(
                                any=parent_asins
                            )
                        )
                    ]
                ),
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )

    retrieved_context_ids = []
    retieved_context = []
    similarity_scores = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retieved_context.append(result.payload["preprocessed_data"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retieved_context,
        "similarity_scores": similarity_scores
    }


def process_reviews_context(context):

    formatted_context = ""

    for id, chunk in zip(context["retrieved_context_ids"], context["retrieved_context"]):
        formatted_context += f"- ID: {id}, description: {chunk}\n"

    return formatted_context


def get_formatted_item_context(query: str, parent_asins: list[str], top_k: int = 5) -> str:

    """Get the top k reviews matching a query for a list of prefiltered items.
    
    Args:
        query: The query to get the top k reviews for
        item_list: The list of item IDs to prefilter for before running the query
        top_k: The number of reviews to retrieve, this should be at least 20 if multipple items are prefiltered
    
    Returns:
        A string of the top k context chunks with IDs prepending each chunk, each representing a review for a given inventory item for a given query.
    """

    context = retrieve_prefiltered_reviews_data(query, parent_asins, top_k)
    formatted_context = process_reviews_context(context)

    return formatted_context